## Homework 1: Sector Deep Dive - Consumer Defensive

**Course:** Big Data Analytics  
**Professor:** Giovanni Morana  
**Student:** Carmen Padova  
**Student ID:** 100003559 (4)

## **Step 0 - Setup and Sector Filter**


In [ ]:
# =========================================================
# STEP 0 — Dataset Setup and Sector Filtering
# =========================================================

# 1. LIBRARY IMPORTS

# pandas: for managing and manipulating tabular datasets
import pandas as pd

# numpy: for advanced mathematical and numerical operations
import numpy as np

# matplotlib and seaborn: for data visualization
import matplotlib.pyplot as plt
import seaborn as sns


# 2. PLOT STYLE SETTINGS

# We use the "whitegrid" theme to ensure charts are readable and consistent with a data analysis report.
sns.set_theme(style="whitegrid")


# 3. DATASET LOADING

# The dataset is loaded into a pandas DataFrame.
df_raw = pd.read_csv('data/symbol_info_3-25.csv')


# 4. DATA CLEANING

# We keep only:
# - companies that are not ETFs;
# - companies that are not funds;
# - companies actively trading on the market;
# - records with non-null values for 'market_cap' and 'sector',
#   as this information is essential for the analysis.

df_cleaned = df_raw[
    (df_raw['is_etf'] == 0) &                # NO ETFs
    (df_raw['is_fund'] == 0) &               # NO FUNDS
    (df_raw['is_actively_trading'] == 1) &   # Keep ACTIVE companies
    (df_raw['market_cap'].notna()) &         # Remove missing Market Cap values
    (df_raw['sector'].notna())               # Remove missing Sector values
].copy()


# 5. SECTOR FILTERING

# We select only the companies belonging to the "Consumer Defensive" sector, the assigned sector for this analysis.

df_sector = df_cleaned[
    df_cleaned['sector'] == 'Consumer Defensive'
].copy()


# 6. RESULTS VERIFICATION

# We print the row count before and after filtering to verify that the cleaning process was successful.

print(f"Total rows in the original dataset: {len(df_raw)}")
print(f"Filtered companies in the 'Consumer Defensive' sector: {len(df_sector)}")


# 7. FINAL DATASET VISUAL INSPECTION

# We display the first few rows of the final dataset to check its structure and available variables.

df_sector.head()

## **STEP 1: MARKET CAPITALIZATION DISTRIBUTION**

In [ ]:
# =========================================================
# STEP 1 — Market Cap Distribution
# =========================================================

# We calculate the mean and median of the market cap.
# Using both allows for a better understanding of the distribution:
# - the mean can be heavily influenced by outliers (extremely large companies);
# - the median represents the central value of the distribution.

mean_val = df_sector['market_cap'].mean()
median_val = df_sector['market_cap'].median()


# Initialize the plot.
plt.figure(figsize=(12, 6))


# Values are divided by 1e9 to display the market cap in billions (B) of dollars,
# making the axis labels and the overall chart more readable.

sns.histplot(
    df_sector['market_cap'] / 1e9,
    bins=12,
    color="skyblue"
)


# Add a vertical dashed line for the mean and a solid line for the median,
# including the calculated values in the legend.

plt.axvline(
    mean_val / 1e9,
    color='red',
    linestyle='--',
    label=f'Mean: ${mean_val/1e9:.2f}B'
)

plt.axvline(
    median_val / 1e9,
    color='green',
    linestyle='-',
    label=f'Median: ${median_val/1e9:.2f}B'
)


# Set the title and axis labels.

plt.title(
    "Market Cap Distribution - Consumer Defensive",
    fontsize=15,
    fontweight='bold'
)

plt.xlabel("Market Cap (Billions of dollars)")
plt.ylabel("Number of Companies")


# Display the legend to show the mean and median indicators.
plt.legend()


# Optimize the layout to prevent overlapping of graphic elements.
plt.tight_layout()

plt.show()

## **STEP 2: OUTLIER DETECTION**

In [ ]:
# =========================================================
# STEP 2 — Outlier Detection
# =========================================================

# We use the IQR (Interquartile Range) method 
# to identify companies with a market cap significantly 
# different from the majority of the dataset.

# 1. CALCULATING QUARTILES AND IQR

Q1 = df_sector['market_cap'].quantile(0.25)
Q3 = df_sector['market_cap'].quantile(0.75)

# The IQR measures the spread of the middle 50% of the data.
IQR = Q3 - Q1


# Define the thresholds used to identify outliers.

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


# 2. IDENTIFYING OUTLIERS

# We consider companies with a market cap 
# higher than the upper bound as outliers.

outliers_df = df_sector[
    df_sector['market_cap'] > upper_bound
].sort_values(by='market_cap', ascending=False).copy()


# 3. PRINTING KEY STATISTICS

print("=" * 80)
print("BOXPLOT - MARKET CAP")
print("=" * 80)

print("STATISTICS:")
print(f"Q1 (25th percentile): ${Q1/1e9:.2f}B")
print(f"Q3 (75th percentile): ${Q3/1e9:.2f}B")
print(f"IQR: ${IQR/1e9:.2f}B")

print("\nOUTLIER THRESHOLDS:")
print(f"Lower Bound: ${lower_bound/1e9:.2f}B")
print(f"Upper Bound: ${upper_bound/1e9:.2f}B")

print("\nRESULTS:")
print(f"Initial companies: {len(df_sector)}")
print(f"Outliers identified: {len(outliers_df)}")


# Display the companies identified as outliers.

print("\nIDENTIFIED OUTLIERS:")

for i, (idx, row) in enumerate(outliers_df.iterrows(), 1):
    symbol = row['symbol']
    mcap = row['market_cap'] / 1e9

    print(f"{i}. {symbol} - ${mcap:.2f}B")


# 4. REMOVING OUTLIERS

# We create a new dataset excluding outliers.
# From this point forward, we will use df_filtered.

df_filtered = df_sector[
    df_sector['market_cap'] <= upper_bound
].copy()


# 5. CREATING THE BOXPLOT

plt.figure(figsize=(12, 7))


# Convert market cap to billions of dollars
# to improve chart readability.

data_to_plot = [
    df_sector["market_cap"].dropna().values / 1e9
]


# Create the distribution boxplot.

plt.boxplot(
    data_to_plot,
    tick_labels=["Consumer Defensive"],
    vert=False,
    patch_artist=True,

    boxprops={
        "facecolor": "#A0CBE8",
        "edgecolor": "#333333"
    },

    medianprops={
        "color": "#E45756",
        "linewidth": 2
    },

    flierprops={
        "marker": "o",
        "markerfacecolor": "gold",
        "markeredgecolor": "#333333",
        "markersize": 7
    }
)


# Add threshold lines for outliers.

plt.axvline(
    lower_bound / 1e9,
    color='purple',
    linestyle='--',
    alpha=0.8,
    label='Lower Bound'
)

plt.axvline(
    upper_bound / 1e9,
    color='red',
    linestyle='--',
    alpha=0.8,
    label='Upper Bound'
)


# Set chart title and axis labels.

plt.title(
    "Market Cap Boxplot - Outlier Identification",
    fontsize=14,
    fontweight='bold'
)

plt.xlabel("Market Cap (Billions of dollars)")


# Display the legend.

plt.legend()

plt.grid(axis="x", alpha=0.2)

plt.tight_layout()

plt.show()


# 6. FINAL CHECK

print("\n" + "=" * 80)
print("CLEANED DATASET")
print("=" * 80)

print(f"Companies removed: {len(outliers_df)}")
print(f"Remaining companies: {len(df_filtered)}")

## **STEP 3: Industry Comparison: Four Metrics**

In [ ]:
# =========================================================
# STEP 3 — Industry Comparison: Four Key Metrics
# =========================================================

# We create 4 charts to compare industries within the 
# cleaned dataset using the following metrics:
# - market cap
# - total revenue
# - profit margins
# - beta

# 1. DEFINING METRICS AND TITLES

metrics = [
    'market_cap',
    'total_revenue',
    'profit_margins',
    'beta'
]

titles = [
    'Mean Market Capitalization',
    'Mean Total Revenue',
    'Mean Profit Margins',
    'Mean Beta'
]


# 2. CREATING CONSISTENT COLOR PALETTE

# We maintain the same color for each industry across all 
# charts to facilitate quick visual comparison.

industries = sorted(
    df_filtered['industry'].dropna().unique()
)

palette = sns.color_palette(
    "tab20", 
    len(industries)
)

industry_colors = dict(
    zip(industries, palette)
)


# 3. SETTING UP FIGURE AND SUBPLOTS

fig, axes = plt.subplots(
    2, 
    2, 
    figsize=(20, 14)
)

# Flatten the axes array to easily iterate over it.

axes = axes.flatten()


# 4. PLOT GENERATION

for i, metric in enumerate(metrics):

    # Calculate the mean value of the metric grouped by industry.

    industry_mean = (
        df_filtered
        .groupby('industry')[metric]
        .mean()
        .dropna()
        .sort_values(ascending=True)
    )


    # Convert specific metrics to improve chart readability.

    if metric in ['market_cap', 'total_revenue']:

        values = industry_mean.values / 1e9
        xlabel = 'Billions of $'

    elif metric == 'profit_margins':

        values = industry_mean.values * 100
        xlabel = 'Profit Margin (%)'

    else:

        values = industry_mean.values
        xlabel = 'Beta'


    # Map colors to each specific industry in the plot.

    colors = [
        industry_colors[ind] 
        for ind in industry_mean.index
    ]


    # Create the horizontal bar chart.

    axes[i].barh(
        industry_mean.index, 
        values, 
        color=colors, 
        edgecolor='black'
    )


    # Chart customization and styling.

    axes[i].set_title(
        titles[i], 
        fontsize=14, 
        fontweight='bold'
    )

    axes[i].set_xlabel(xlabel)
    axes[i].set_ylabel('Industry')

    axes[i].grid(
        axis='x', 
        alpha=0.3
    )

    # Remove top and right spines for a cleaner aesthetic.

    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)


# 5. OVERALL FIGURE TITLE

fig.suptitle(
    'Industry Comparison Across Four Metrics', 
    fontsize=20, 
    fontweight='bold'
)


# 6. LAYOUT OPTIMIZATION

plt.tight_layout()

plt.show()

## **STEP 4: Industry Dispersion**

In [ ]:
# =========================================================
# STEP 4 — Industry Dispersion Analysis
# =========================================================

# We create 4 boxplots to observe the full distribution
# of metrics within each industry.
#
# Unlike the previous step, here we analyze not only the 
# mean but also the dispersion and variance of the values.

metrics_cfg = [
    ('market_cap', 'Market Capitalization ($B)', 1e9),
    ('total_revenue', 'Total Revenue ($B)', 1e9),
    ('profit_margins', 'Profit Margins (%)', 0.01),
    ('beta', 'Beta', 1)
]


# Initialize a figure with 4 subplots.

fig, axes = plt.subplots(
    2, 
    2, 
    figsize=(22, 24)
)

axes = axes.flatten()


# Generate a boxplot for each metric.

for i, (col, title, divisor) in enumerate(metrics_cfg):

    # Group data by industry for the current metric.

    data_to_plot = [
        df_filtered[
            df_filtered['industry'] == ind
        ][col].dropna().values / divisor 
        
        for ind in industries
    ]


    # Create the horizontal boxplot.

    bplot = axes[i].boxplot(
        data_to_plot, 
        tick_labels=industries, 
        vert=False, 
        patch_artist=True,

        medianprops={
            "color": "black", 
            "linewidth": 2
        },

        flierprops={
            "marker": "d", 
            "markerfacecolor": "gray", 
            "markersize": 4, 
            "alpha": 0.4
        },

        widths=0.7
    )


    # Apply consistent coloring across all charts.

    for patch, ind_name in zip(bplot['boxes'], industries):

        patch.set_facecolor(
            industry_colors[ind_name]
        )

        patch.set_edgecolor('black')


    # Set chart titles and grid styling.

    axes[i].set_title(
        title, 
        fontsize=18, 
        fontweight='bold'
    )

    axes[i].grid(
        axis='x', 
        linestyle=':', 
        alpha=0.6
    )


    # Remove top and right spines for a cleaner look.

    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)


# Overall figure title.

fig.suptitle(
    'Consumer Defensive: Distribution by Industry', 
    fontsize=26, 
    fontweight='bold', 
    y=0.98
)


# Optimize spacing between subplots.

plt.tight_layout(
    rect=[0, 0.03, 1, 0.96], 
    h_pad=10.0
)

plt.show()

## **STEP 5: Focus Industry Selection**

In [ ]:
# =========================================================
# STEP 5 — Industry Selection
# =========================================================

# Based on the results from Steps 3 and 4, 
# we select 4 industries with distinct financial 
# characteristics to proceed with the analysis.

selected_industries = [
    'Grocery Stores',
    'Confectioners',
    'Beverages - Brewers',
    'Packaged Foods'
]


# Filter the dataset to keep only 
# the selected industries.

df_final_selection = df_filtered[
    df_filtered['industry'].isin(selected_industries)
].copy()


# Final dataset verification.

print(f"Number of selected companies: {len(df_final_selection)}")

print(
    f"Industries retained: "
    f"{df_final_selection['industry'].unique().tolist()}"
)

In [ ]:
# Basandoci sull'analisi delle medie (Step 3) e della dispersione (Step 4), 
# abbiamo selezionato 4 industry che rappresentano modelli finanziari differenti.

# Creazione della lista basata sui grafici visualizzati
selected_industries = [
    'Grocery Stores', 
    'Confectioners', 
    'Beverages - Brewers', 
    'Packaged Foods'
]

# Filtraggio del dataset
df_final_selection = df_filtered[df_filtered['industry'].isin(selected_industries)].copy()

# Verifica veloce
print(f"Dataset finale pronto con {len(df_final_selection)} aziende.")
print(f"Settori mantenuti: {df_final_selection['industry'].unique().tolist()}")


## Justifications of the choises 

**INDUSTRIES SELECTED**
# STEP 5 — Industry Selection

**INDUSTRIES SELECTED**

***Grocery Stores***

Selected because it shows high average revenue and a relatively compact distribution in the boxplots, suggesting more stable performance compared to other industries.


***Confectioners***

Selected because it has lower market capitalization but more concentrated profit margins, making it useful for comparison with larger industries.


***Beverages - Brewers***

Selected because it shows high market capitalization and a wider distribution, indicating the presence of both medium and very large companies.


***Packaged Foods***

Selected because it presents balanced values across most metrics and moderate dispersion in the boxplots.

**INDUSTRIES EXCLUDED**

***With very few observations*** were excluded because the distributions were not sufficiently representative.

***With extremely high dispersion*** were excluded because comparisons would be heavily influenced by a small number of very large companies.

## **STEP 6: Bubble Chartù**

In [ ]:
# =========================================================
# STEP 6 — Strategic Positioning: Bubble Chart Analysis
# =========================================================

# TEACHING POINTS & VISUAL ENCODING RATIO:
# This chart integrates multiple financial dimensions to evaluate the 
# competitive positioning of the selected companies:
# - X-Axis (Revenue Growth): Measures the company's ability to expand its market share.
# - Y-Axis (Profit Margins): Highlights operational efficiency and profitability.
# - Bubble Size (Total Revenue): Scales the business size, providing context 
#   on whether growth/margins are coming from a large corporation or a smaller player.
# - Bubble Color (Beta): Visualizes market risk/volatility (Red/Coolwarm palette).
# - Bubble Edge (Industry): Distinct colors per industry to identify sector clusters.

plt.figure(figsize=(18, 12))

# 1. VARIABLE DEFINITION & SCALING
# Converting decimals to percentages for better readability on axes.
x = df_final_selection['revenue_growth'] * 100
y = df_final_selection['profit_margins'] * 100

# Total revenue in Billions is used for bubble scaling.
bubble_size = (df_final_selection['total_revenue'] / 1e9) * 80
risk_color = df_final_selection['beta']
tickers = df_final_selection['symbol']

# Mapping industry-specific colors to the bubble edges for visual grouping.
edge_colors = [industry_colors[ind] for ind in df_final_selection['industry']]

# 2. BUBBLE CHART EXECUTION
scatter = plt.scatter(
    x, 
    y, 
    s=bubble_size, 
    c=risk_color, 
    cmap='coolwarm', 
    alpha=0.45, 
    edgecolors=edge_colors, 
    linewidth=2.5           # Increased width to emphasize industry color
)

# 3. COLORBAR FOR RISK ASSESSMENT (BETA)
cbar = plt.colorbar(scatter)
cbar.set_label('Beta (Market Risk Level)', fontsize=11)

# 4. INDUSTRY CUSTOM LEGEND
# Since edge colors represent industries, we create a proxy legend.
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', label=ind,
                          markerfacecolor='none', markeredgecolor=color, 
                          markersize=10, markeredgewidth=2)
                   for ind, color in industry_colors.items() if ind in selected_industries]

plt.legend(handles=legend_elements, title="Industries", loc="upper left", bbox_to_anchor=(1.15, 1))

# 5. DATA ANNOTATION (TICKERS)
# Labeling each bubble to pinpoint specific company performance.
for i in range(len(df_final_selection)):
    plt.text(
        x.iloc[i], 
        y.iloc[i] + 0.2, 
        tickers.iloc[i], 
        fontsize=8, 
        ha='center',
        fontweight='bold'
    )

# 6. QUADRANT DEFINITION & INTERPRETATION
# Quadrants are defined by a vertical line at 0% growth and 
# a horizontal line at the mean profit margin.
plt.axvline(0, color='black', linewidth=1, alpha=0.3)
plt.axhline(y.mean(), color='black', linestyle='--', alpha=0.3)

# Quadrant Labels (Color Coded by Strategic Meaning):
# High Growth/High Margin: Strongest performers (Top Right).
plt.text(x.max()*0.7, y.max()*0.9, 'High Growth\nHigh Margin', fontsize=12, fontweight='bold', color='forestgreen')
# Low Growth/High Margin: Mature/Cash Cow companies (Top Left).
plt.text(x.min()*0.9, y.max()*0.9, 'Low Growth\nHigh Margin', fontsize=12, fontweight='bold', color='royalblue')
# Low Growth/Low Margin: Companies in distress or stagnant sectors (Bottom Left).
plt.text(x.min()*0.9, y.min()*1.1, 'Low Growth\nLow Margin', fontsize=12, fontweight='bold', color='crimson')
# High Growth/Low Margin: Rapid expansion but low efficiency (Bottom Right).
plt.text(x.max()*0.7, y.min()*1.1, 'High Growth\nLow Margin', fontsize=12, fontweight='bold', color='darkorange')

# 7. FINAL STYLING
plt.title("Consumer Defensive: Strategic Positioning & Industry Analysis", fontsize=20, fontweight='bold')
plt.xlabel("Revenue Growth (%)")
plt.ylabel("Profit Margin (%)")

plt.grid(True, linestyle=':', alpha=0.3)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## **STEP 7: Zoom into One Quadrant**

In [ ]:
# =========================================================
# STEP 7 — Zoom into One Quadrant
# =========================================================

# Chosen Quadrant:
# Low Growth / High Margin
#
# This quadrant was selected because it includes companies 
# with high margins but more contained growth. 
# This allows us to observe whether companies perceived 
# as more stable experienced significant effects during 
# the geopolitical tensions in the Strait of Hormuz.

import yfinance as yf
import matplotlib.pyplot as plt


# 1. QUADRANT FILTERING

# We select:
# - Companies with revenue growth less than or equal to 0;
# - Companies with profit margins above the average.

quadrant_df = df_final_selection[
    (df_final_selection['revenue_growth'] <= 0) & 
    (
        df_final_selection['profit_margins'] 
        > df_final_selection['profit_margins'].mean()
    )
].copy()


# 2. COMPANY SELECTION

# If the quadrant contains more than 10 companies, 
# we keep the ones with the highest total revenue.

quadrant_df = quadrant_df.sort_values(
    by='total_revenue', 
    ascending=False
).head(10)


# Extract the tickers from the dataset.

tickers_list = quadrant_df['symbol'].tolist()


# Remove any tickers that might not return 
# correct data from Yahoo Finance.

tickers_list = [
    ticker for ticker in tickers_list 
    if ticker != 'K'
]


print(f"Selected Tickers: {tickers_list}")


# 3. STOCK PRICE DOWNLOAD

# We download the closing prices for the last 3 months.

df_prices = yf.download(
    tickers_list, 
    period="3mo"
)['Close']


# 4. CLEANING AND NORMALIZATION

# Remove any completely empty columns.

df_prices = df_prices.dropna(
    axis=1, 
    how='all'
)


# Normalize all time series to 1 
# based on the first available day.

df_norm = df_prices / df_prices.iloc[0]


# 5. CHART CREATION

plt.figure(figsize=(16, 8))


# Plot the normalized time series.

for ticker in df_norm.columns:

    plt.plot(
        df_norm.index, 
        df_norm[ticker], 
        label=ticker, 
        linewidth=2
    )


# Baseline reference line.

plt.axhline(
    1, 
    color='black', 
    linestyle='--', 
    alpha=0.5
)


# Title and axis labels.

plt.title(
    "Normalized Stock Prices - Last 3 Months", 
    fontsize=18, 
    fontweight='bold'
)

plt.xlabel("Date")
plt.ylabel("Normalized Price")


# Display the legend.

plt.legend(
    title="Companies", 
    bbox_to_anchor=(1, 1)
)


# Add grid for better readability.

plt.grid(
    True, 
    linestyle=':', 
    alpha=0.4
)

plt.tight_layout()

plt.show()

### Interpretation

Companies belonging to the “Low Growth / High Margin” quadrant do not show an extremely negative impact during the period of geopolitical tensions in the Strait of Hormuz. No sharp or sudden crashes are observed; instead, the overall trend is relatively stable, with some phases of gradual decline.

**GIS** shows the most evident decrease during the observed period, with prices progressively falling well below the initial normalized value. **TAP** also records a negative trend, although less pronounced. **KHC** experiences a weaker phase in the middle of the chart, followed by a moderate partial recovery.

**INGR**, on the other hand, appears to be one of the most stable companies, remaining relatively close to its initial value for most of the period. This behavior suggests that companies characterized by high margins may be perceived as more defensive during periods of economic and geopolitical uncertainty.

Overall, the chart suggests that the tensions in the Strait of Hormuz had a moderate impact on the selected companies, without causing sharp shocks or widespread collapses.


## **STEP 8: Compare with an Excluded Quadrant**


In [ ]:
# =========================================================
# STEP 8 — Comparison with an Excluded Quadrant
# =========================================================

# Chosen Quadrant:
# Low Growth / Low Margin
#
# This quadrant was selected to compare its performance 
# with the one analyzed in Step 7.
# The objective is to verify whether companies with lower 
# margins reacted differently to the geopolitical 
# tensions in the Strait of Hormuz.

import yfinance as yf
import matplotlib.pyplot as plt


# 1. QUADRANT FILTERING

# We select:
# - Companies with revenue growth less than or equal to 0;
# - Companies with profit margins below or equal to the average.

quadrant_under = df_final_selection[
    (df_final_selection['revenue_growth'] <= 0) & 
    (
        df_final_selection['profit_margins'] 
        <= df_final_selection['profit_margins'].mean()
    )
].copy()


# 2. COMPANY SELECTION

# We use the same metric as in Step 7 for consistency:
# total revenue.

quadrant_under = quadrant_under.sort_values(
    by='total_revenue', 
    ascending=False
).head(10)


# Extract the tickers.

tickers_under = quadrant_under['symbol'].tolist()


# Remove any problematic tickers.

tickers_under = [
    ticker for ticker in tickers_under 
    if ticker != 'K'
]


print(f"Selected Tickers: {tickers_under}")


# 3. STOCK PRICE DOWNLOAD

df_prices_under = yf.download(
    tickers_under, 
    period="3mo"
)['Close']


# 4. CLEANING AND NORMALIZATION

df_prices_under = df_prices_under.dropna(
    axis=1, 
    how='all'
)

df_norm_under = (
    df_prices_under / 
    df_prices_under.iloc[0]
)


# 5. CHART CREATION

plt.figure(figsize=(16, 8))


for ticker in df_norm_under.columns:

    plt.plot(
        df_norm_under.index, 
        df_norm_under[ticker], 
        label=ticker, 
        linewidth=2
    )


# Baseline reference line.

plt.axhline(
    1, 
    color='black', 
    linestyle='--', 
    alpha=0.5
)


# Title and axis labels.

plt.title(
    "Normalized Stock Prices - Low Growth / Low Margin", 
    fontsize=18, 
    fontweight='bold'
)

plt.xlabel("Date")
plt.ylabel("Normalized Price")


# Legend.

plt.legend(
    title="Companies", 
    bbox_to_anchor=(1, 1)
)


# Grid.

plt.grid(
    True, 
    linestyle=':', 
    alpha=0.4
)

plt.tight_layout()

plt.show()

### Analysis of the Low Growth / Low Margin Group

The companies in the “Low Growth / Low Margin” quadrant show weaker and more volatile performance over the last three months. Several firms experienced gradual declines below the initial normalized value, while only a few remained relatively stable during the observed period.

### Comparison with Step 7

Compared to the “Low Growth / High Margin” group analyzed in Step 7, these companies appear less resilient and more exposed to negative market movements. The companies from Step 7 generally showed more stable performance and smaller declines, while this group displayed greater volatility and weaker recovery patterns.

### Geopolitical Impact (Strait of Hormuz)

The geopolitical tensions in the Strait of Hormuz appear to have had a more visible impact on this group. Some companies experienced temporary declines and larger fluctuations, suggesting higher sensitivity to uncertainty and market stress. However, the chart does not show evidence of a sharp collapse across the entire group.

# **FINAL SUMMARY**

This analysis explored the structure of the Consumer Defensive sector using different visualizations to compare company size, profitability, growth, and risk. The initial distribution of market capitalization showed the presence of a few very large companies alongside many smaller firms, confirming that the sector is not homogeneous.

The comparison between industries highlighted important differences across the sector. Grocery Stores showed high average revenue and relatively stable distributions, while Confectioners presented smaller companies with more concentrated margins. Beverages - Brewers showed wider dispersion and higher variability, whereas Packaged Foods appeared more balanced across most metrics. These industries were selected because they represented different financial profiles within the sector.

The bubble chart combined growth, profitability, company size, and risk into a single visualization. This made it easier to identify different groups of companies and compare their positioning. Companies with higher profit margins generally appeared more stable, while lower-margin firms often showed weaker performance and greater volatility.

The comparison between quadrants also revealed different market behaviors during the last three months. Companies belonging to the “Low Growth / High Margin” quadrant showed relatively stable performance despite the geopolitical tensions in the Strait of Hormuz. Most companies experienced moderate declines but remained fairly resilient overall.

In contrast, the “Low Growth / Low Margin” group appeared more exposed to negative market movements. Several companies showed more persistent declines and weaker recovery signals. While no extreme crashes occurred, this group generally displayed higher volatility and lower stability.

Overall, the analysis suggests that companies with stronger profitability tend to be more resilient during periods of geopolitical uncertainty, while firms with weaker margins appear more vulnerable to market fluctuations.